# Bootstrapping, Bagging y Ensemble Models
**Dataset:** Motor Trend Car Road Tests  
**Objetivo:** Predecir `mpg` usando técnicas de remuestreo y ensamble

* Francisco Tinoco

In [1]:
pip install scikit-optimize

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

import statsmodels.api as sm
from skopt import BayesSearchCV

---
## Carga de datos

In [3]:
df=pd.read_excel('../Data/Motor Trend Car Road Tests.xlsx')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   model   32 non-null     object 
 1   mpg     32 non-null     float64
 2   cyl     32 non-null     int64  
 3   disp    32 non-null     float64
 4   hp      32 non-null     int64  
 5   drat    32 non-null     float64
 6   wt      32 non-null     float64
 7   qsec    32 non-null     float64
 8   vs      32 non-null     int64  
 9   am      32 non-null     int64  
 10  gear    32 non-null     int64  
 11  carb    32 non-null     int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 3.1+ KB


In [4]:
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [3]:
df.columns

Index(['model', 'mpg', 'cyl', 'disp', 'hp', 'drat', 'wt', 'qsec', 'vs', 'am',
       'gear', 'carb'],
      dtype='object')

---
##  Bootstrapping
Estimamos los coeficientes β de la regresión `mpg = β0 + β1(hp) + β2(qsec)` con OLS, y luego validamos esos intervalos de confianza usando bootstrap con 1000 muestras con reemplazo.

### 2a. Regresión OLS original

In [4]:
X = df[['hp', 'qsec']]
y = df['mpg']

X_sm = sm.add_constant(X)
model_fit = sm.OLS(y, X_sm).fit()
print(model_fit.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 04 May 2026   Prob (F-statistic):           4.18e-07
Time:                        18:09:22   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

In [5]:
print('Intervalos de confianza (OLS):')
print(model_fit.conf_int())

Intervalos de confianza (OLS):
               0          1
const  25.614894  71.032516
hp     -0.113089  -0.056097
qsec   -1.979929   0.206770


### 2b. Bootstrap — 1000 muestras con reemplazo

In [7]:
n_bootstrap = 1000
m = len(df)
betas_bootstrap = []

for _ in range(n_bootstrap):
    # Muestra con reemplazo del mismo tamaño que el dataset
    muestra = df.sample(n=m, replace=True)

    X_muestra = sm.add_constant(muestra[['hp', 'qsec']])
    y_muestra = muestra['mpg']

    modelo_b = sm.OLS(y_muestra, X_muestra).fit()
    betas_bootstrap.append(modelo_b.params)

betas_df = pd.DataFrame(betas_bootstrap, columns=['B0', 'B1', 'B2'])

### 2c. Comparación OLS vs Bootstrap

In [8]:
medias    = betas_df.mean()
desv_std  = betas_df.std()
ic_boot   = betas_df.quantile([0.025, 0.975])
ic_ols    = model_fit.conf_int()

comparacion = pd.DataFrame({
    'OLS β':           model_fit.params.values,
    'OLS IC_low':      ic_ols[0].values,
    'OLS IC_high':     ic_ols[1].values,
    'Boot β̄':         medias.values,
    'Boot σ':          desv_std.values,
    'Boot IC_low':     ic_boot.loc[0.025].values,
    'Boot IC_high':    ic_boot.loc[0.975].values,
}, index=['B0', 'B1', 'B2'])


In [9]:
comparacion

,OLS β,OLS IC_low,OLS IC_high,Boot β̄,Boot σ,Boot IC_low,Boot IC_high
B0,48.323705,25.614894,71.032516,NaN,NaN,NaN,NaN
B1,-0.084593,-0.113089,-0.056097,NaN,NaN,NaN,NaN
B2,-0.886580,-1.979929,0.206770,NaN,NaN,NaN,NaN


---
## Aggregating (Bagging manual)
Entrenamos 1000 regresiones lineales, cada una con 3 variables aleatorias distintas. El pronóstico final es el **promedio** de todos los modelos.

In [10]:
X_all = df.drop(['mpg', 'model'], axis=1)
y_all = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.5, random_state=42
)

In [11]:
columnas = list(X_all.columns)
modelos, vars_por_iter = [], []

for _ in range(1000):
    # Selección aleatoria de 3 variables sin reemplazo
    vars_i = np.random.choice(columnas, size=3, replace=False)
    modelo_i = LinearRegression().fit(X_train[vars_i], y_train)

    modelos.append(modelo_i)
    vars_por_iter.append(vars_i)


In [ ]:
predicciones = [
    modelos[i].predict(X_test[vars_por_iter[i]])
    for i in range(1000)
]

y_hat = np.array(predicciones).mean(axis=0)
r2_bagging = r2_score(y_test, y_hat)
r2_bagging

0.7842150618367215

---
## Random Forest
Modelo de ensamble basado en árboles de decisión con muestreo aleatorio de features. Incluye validación cruzada y ajuste de hiperparámetros.

In [14]:
rf_base = RandomForestRegressor(random_state=42)
rf_base.fit(X_all, y_all)

r2_rf_base = r2_score(y_all, rf_base.predict(X_all))
r2_rf_base

0.9770244219894207

### 4a. Validación cruzada K-Fold (k=10)

In [15]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)

cv_r2  = cross_val_score(rf_base, X_all, y_all, cv=cv, scoring='r2')
cv_mse = cross_val_score(rf_base, X_all, y_all, cv=cv, scoring='neg_mean_squared_error')

print(f"Mean R²  (CV): {cv_r2.mean():.4f}")
print(f"Mean MSE (CV): {-cv_mse.mean():.4f}")

Mean R²  (CV): 0.4552
Mean MSE (CV): 5.9575


### 4b. Hyperparameter Tuning — Bayesian Search
Con un dataset pequeño, ajustar los hiperparámetros es importante para evitar overfitting.

In [16]:
search_space = {
    'n_estimators':      (3, 100),
    'max_depth':         (2, 10),
    'min_samples_split': (2, 10),
    'min_samples_leaf':  (1, 5),
    'max_features':      ['sqrt', 'log2', None]
}

opt = BayesSearchCV(
    RandomForestRegressor(random_state=42),
    search_space,
    n_iter=32, cv=10, n_jobs=-1,
    scoring='r2', random_state=42
)
opt.fit(X_all, y_all)

print(f"Mejor R² (Bayes): {opt.best_score_:.4f}")
print(f"Mejores params:   {opt.best_params_}")

Mejor R² (Bayes): -2.4105
Mejores params:   OrderedDict([('max_depth', 8), ('max_features', None), ('min_samples_leaf', 2), ('min_samples_split', 4), ('n_estimators', 55)])


### 4c. Hyperparameter Tuning — Grid Search

In [17]:
gs = GridSearchCV(
    rf_base,
    param_grid={
        'max_depth':         range(1, 11),
        'min_samples_split': range(10, 60, 10),
        'n_estimators':      range(1, 100, 10)
    },
    cv=10, scoring='r2'
)
gs.fit(X_all, y_all)

print(f"Mejor R² (Grid Search): {gs.best_score_:.4f}")
print(f"Mejores params:         {gs.best_params_}")

Mejor R² (Grid Search): -4.0815
Mejores params:         {'max_depth': 4, 'min_samples_split': 10, 'n_estimators': 51}


### 4d. Modelo final con parámetros óptimos

In [18]:
rf_final = RandomForestRegressor(
    n_estimators=51,
    criterion='squared_error',
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=1,
    bootstrap=True,
    oob_score=False,
    random_state=42
)
rf_final.fit(X_all, y_all)

r2_rf_final = r2_score(y_all, rf_final.predict(X_all))
print(f"R² Random Forest (tuned): {r2_rf_final:.4f}")

R² Random Forest (tuned): 0.9078


---
##  Boosting (Gradient Boosting)
A diferencia del bagging, los árboles se construyen **secuencialmente**, corrigiendo los errores del anterior.

In [19]:
gb_base = GradientBoostingRegressor(random_state=42)
gb_base.fit(X_all, y_all)

r2_gb_base = r2_score(y_all, gb_base.predict(X_all))
r2_gb_base

0.9999297755665103

In [20]:
gb_base.get_params()

{'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 42,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

### 5a. Validación cruzada K-Fold

In [21]:
cv_r2_gb  = cross_val_score(gb_base, X_all, y_all, cv=cv, scoring='r2')
cv_mse_gb = cross_val_score(gb_base, X_all, y_all, cv=cv, scoring='neg_mean_squared_error')

print(f"Mean R²  (CV): {cv_r2_gb.mean():.4f}")
print(f"Mean MSE (CV): {-cv_mse_gb.mean():.4f}")

Mean R²  (CV): 0.5640
Mean MSE (CV): 6.4218


### 5b. Grid Search — Gradient Boosting

In [22]:
param_grid_gb = {
    'n_estimators':  [20, 50, 100],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth':     [2, 3],
    'subsample':     [0.6, 0.8, 1.0],
    'max_features':  ['sqrt', None]
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid=param_grid_gb,
    cv=5, scoring='r2', n_jobs=-1
)
gb_grid.fit(X_all, y_all)

print(f"Mejor R² (Grid Search): {gb_grid.best_score_:.4f}")
print(f"Mejores params:         {gb_grid.best_params_}")

Mejor R² (Grid Search): 0.7421
Mejores params:         {'learning_rate': 0.1, 'max_depth': 2, 'max_features': 'sqrt', 'n_estimators': 100, 'subsample': 0.8}


### 5c. Modelo final con parámetros óptimos

In [23]:
gb_final = GradientBoostingRegressor(
    criterion='squared_error',
    max_depth=2,
    max_features='sqrt',
    min_samples_leaf=1,
    random_state=42,
    subsample=0.8
)
gb_final.fit(X_all, y_all)

r2_gb_final = r2_score(y_all, gb_final.predict(X_all))
print(f"R² Gradient Boosting (tuned): {r2_gb_final:.4f}")

R² Gradient Boosting (tuned): 0.9955


---
## Resumen de Resultados
Comparación del R² de todos los modelos entrenados en este notebook.

In [26]:
resumen = pd.DataFrame({
    'Modelo':  ['Bagging (1000 LR)', 'Random Forest (base)', 'Random Forest (tuned)',
                'Gradient Boosting (base)', 'Gradient Boosting (tuned)'],
    'R²':      [r2_bagging, r2_rf_base, r2_rf_final, r2_gb_base, r2_gb_final]
})

resumen['R²'] = resumen['R²'].round(4)
resumen = resumen.sort_values('R²', ascending=False).reset_index(drop=True)
resumen

,Modelo,R²
0,Gradient Boosting (base),0.9999
1,Gradient Boosting (tuned),0.9955
2,Random Forest (base),0.9770
3,Random Forest (tuned),0.9078
4,Bagging (1000 LR),0.7842
